In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition
#  Adapted from the Kannada-MNIST version.
#
#  Transfer rationale
#  ──────────────────
#  Kuzushiji (くずし字) is classical cursive Japanese script used in pre-Meiji
#  documents.  Kuzushiji-49 covers 49 Hiragana character classes rendered in
#  this historical cursive style.  Key structural differences vs. Kannada:
#
#    • num_classes   : 10  → 49   (Kuzushiji-49: 49 Hiragana classes)
#    • image_size    : 28  → 28   (same – Kuzushiji-49 is natively 28×28)
#    • Dataset format: .npz arrays  (identical layout to Kannada-MNIST / MNIST)
#                      keys: arr_0 (images, uint8), arr_1 (labels, uint8)
#    • Dataset loader: npz loader retained from Kannada version (no changes)
#    • Scaffold branch: 1×3 (Kannada short horizontal) → 3×3 + 3×1 pair
#                       Kuzushiji characters combine flowing diagonal curves
#                       (captured by 3×3 isotropic) with explicit vertical
#                       stroke endings (captured by 3×1).  The horizontal 1×3
#                       is insufficient because Hiragana strokes are often
#                       diagonal, not axis-aligned.
#    • STM kernels   : 1×3/3×1 → 3×3 + 3×1 + two dilated rates (1 and 3)
#                       • 3×3 standard  – diagonal curves and stroke junctions
#                       • 3×1 vertical  – stroke endings / harai (払い) tails
#                       • dilation=1    – local curvature / stroke width
#                       • dilation=3    – character-level spatial context;
#                         Kuzushiji glyphs span the full 28-px frame so wider
#                         context is needed to separate visually similar chars
#                         (e.g. く vs. つ, き vs. さ)
#    • Augmentation  : random_flip REMOVED entirely – Kuzushiji characters
#                       have asymmetric strokes; flipping creates invalid glyphs
#                       (e.g. mirroring き would not produce any valid kana).
#                       random_rotation ±8° ADDED (via tfa or manual affine) –
#                       cursive documents vary greatly in writing angle, so
#                       mild rotation is the most important generalisation.
#                       NOTE: if tensorflow-addons is unavailable, rotation
#                       falls back to pad-crop only (a warning is printed).
#    • Model name    : WhatNet-Kannada → WhatNet-Kuzushiji
#    • Output files  : kannada_*.json  → kuzushiji_*.json
#
#  Everything else – architecture depth, LR schedule, callbacks,
#  display utilities – is unchanged.
#
#  Dataset
#  ───────
#  Kuzushiji-49  (Clanuwat et al., 2018  –  arXiv:1812.01718)
#    • 232 365 train + 38 547 test 28×28 greyscale images
#    • 49 classes (balanced-ish Hiragana subset of full Kuzushiji dataset)
#    • Download:
#        https://github.com/rois-codh/kmnist   (official – npz files)
#        https://www.kaggle.com/datasets/anokas/kuzushiji  (Kaggle mirror)
#    • Files expected in CFG['data_dir']:
#        k49-train-imgs.npz  –  arr_0: uint8 (N, 28, 28)
#        k49-train-lbls.npz  –  arr_0: uint8 (N,)
#        k49-test-imgs.npz   –  arr_0: uint8 (M, 28, 28)
#        k49-test-lbls.npz   –  arr_0: uint8 (M,)
#    • Alternative single-file layout (some mirrors):
#        k49_train.npz       –  keys: arr_0 (images), arr_1 (labels)
#        k49_test.npz        –  keys: arr_0 (images), arr_1 (labels)
#    • This script handles BOTH layouts automatically.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Kannada version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 49 classes – Kuzushiji Hiragana subset
    "num_classes":      49,
    # Same as Kannada-MNIST – Kuzushiji-49 is natively 28×28
    "image_size":       28,

    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # ← CHANGED: path to the Kuzushiji-49 .npz files.
    #   Official download (place *.npz files here):
    #     https://github.com/rois-codh/kmnist
    #   Kaggle mirror:
    #     https://www.kaggle.com/datasets/anokas/kuzushiji
    #   Kaggle notebook path:
    #     "/kaggle/input/kuzushiji"
    "data_dir":    "/kaggle/input/datasets/anokas/kuzushiji/",                          # ← CHANGED

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     ← CHANGED: Kuzushiji-49 ships as .npz files like Kannada-MNIST but uses
#     a SPLIT-FILE layout (separate image and label npz files) OR a combined
#     layout depending on the mirror used.  Both are handled automatically.
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download Kuzushiji-49 from:\n"
        "  https://github.com/rois-codh/kmnist  (official)\n"
        "  https://www.kaggle.com/datasets/anokas/kuzushiji  (Kaggle)\n"
        "Place the .npz files in the directory above."
    )

def _load_split_files(img_path: str, lbl_path: str):
    """
    Load from the official split-file layout:
      k49-train-imgs.npz  (arr_0 = uint8 images)
      k49-train-lbls.npz  (arr_0 = uint8 labels)
    """
    images = np.load(img_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(lbl_path)["arr_0"].astype(np.int32)    # (N,)
    return images, labels

def _load_combined_file(path: str):
    """
    Load from a single combined .npz:
      arr_0 = images (uint8), arr_1 = labels (uint8)
    """
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int32)
    return images, labels

def _load_kuzushiji(split: str):
    """
    Auto-detect layout (split-file vs combined) and load accordingly.
    split: 'train' or 'test'
    """
    d = CFG["data_dir"]
    # ── Layout A: official split files ──
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files: {img_path}, {lbl_path}")
        return _load_split_files(img_path, lbl_path)
    # ── Layout B: combined single file ──
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file: {combined}")
        return _load_combined_file(combined)
    # ── Not found ──
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data for Kuzushiji-49.\n"
        f"  Looked for (split-file): {img_path}, {lbl_path}\n"
        f"  Looked for (combined)  : {combined}\n"
        "Please verify the .npz files are in CFG['data_dir']."
    )

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes in dataset : {len(np.unique(y_train_all))}")
print(f"[INFO] CFG num_classes           : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,28,28) → (N,28,28,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Optional: mild rotation augmentation via tf.keras.layers.RandomRotation ───
# RandomRotation was introduced in TF 2.6. We detect it at runtime to stay
# compatible with older TF versions.
_HAS_RANDOM_ROTATION = hasattr(tf.keras.layers, "RandomRotation")
if _HAS_RANDOM_ROTATION:
    print("[INFO] RandomRotation available – rotation augmentation enabled (±8°).")
    _rotator = tf.keras.layers.RandomRotation(
        factor=8/360,          # ±8 degrees expressed as fraction of 2π
        fill_mode="constant",
        fill_value=-1.0,       # background fill after normalisation
        seed=SEED,
    )
else:
    print("[WARN] RandomRotation not available in this TF version – "
          "rotation augmentation disabled, using pad-crop only.")
    _rotator = None

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Stochastic augmentation – training only.

    Kuzushiji characters come from scanned historical documents with
    significant variation in:
      • Writing angle (cursive script written at various tilts)
      • Ink density / contrast
      • Slight spatial translation

    Augmentations:
      • Brightness / contrast jitter  – ink density variation
      • Pad-then-random-crop          – translation ±2 px
      • Random rotation ±8°           – writing angle variation  ← CHANGED
                                        (no flip – kana are asymmetric)

    No horizontal/vertical flip: mirroring any Kuzushiji glyph produces
    invalid or confusable characters (e.g. く ↔ mirrored-く which does not
    exist; or confusion between き and a non-standard form of さ).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    # ← CHANGED: apply mild rotation if available
    if _rotator is not None:
        img = _rotator(tf.expand_dims(img, 0))[0]
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Kuzushiji stroke topology encoder.                        ← CHANGED docstring

    Kuzushiji-49 (cursive historical Hiragana) has extremely diverse stroke
    patterns:
      • Flowing diagonal curves connecting multiple sub-strokes into one glyph
      • Explicit stroke endings (harai 払い – rightward fading diagonal tails)
      • High inter-class visual similarity: く/つ, き/さ, ぬ/め etc.
      • Glyph strokes span the full 28-px frame → wide spatial context needed

    Kernel design:
      • 3×3 standard    – diagonal curves, stroke junctions, general body
      • 3×1 vertical    – harai tail endpoints, descenders
      • dilation=1      – fine stroke width / curvature (local context)
      • dilation=3      – character-level context: spans ~13 px at 28-px
                           resolution, covering the full glyph height/width;
                           critical for resolving visually similar pairs

    No 1×N wide horizontal kernel: Kuzushiji strokes rarely run purely
    horizontal – diagonal is far more common.  A 3×3 base captures diagonal
    geometry better than axis-aligned asymmetric kernels.
    """
    # ← CHANGED: 3×3 + 3×1 + two dilation rates (1 and 3)
    base = layers.Conv2D(64, 3, padding="same", use_bias=False, activation=gelu)(x)
    v    = layers.Conv2D(32, (3, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1   = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d3   = layers.Conv2D(16, 3, padding="same", dilation_rate=3, use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([base, v, d1, d3])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed WhatNet-Kuzushiji; num_classes=49; image_size=28.
#     Scaffold uses 3×3 isotropic for diagonal Kuzushiji stroke geometry.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_kuzushiji(num_classes: int = 49, image_size: int = 28) -> Model:
    """
    WhatNet-Kuzushiji: WhatNet adapted for Kuzushiji-49 Hiragana recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (glyph body)
      • Diagonal-stroke scaffold: 3×3 isotropic conv – captures diagonal
        cursive strokes that Kuzushiji glyphs are composed of (no axis-aligned
        headline like Bengali/Devanagari, no short cross-hair like Kannada)
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (14×14 at 28-px input)
      enc2: 64→128   ( 7× 7)
      enc3: 128→256  ( 4× 4, with padding)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB) – especially important here
        because fine strokes (d1) and global glyph shape (d3) need to interact
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 3×3 + 3×1 + dilation (rate 1 & 3)
      • Gated fusion → final MLP + layer norm → logits (49 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×3 (Kannada) → 3×3 (Kuzushiji diagonal strokes)
    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(7)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    # ← CHANGED pool factor: 8 → 7  (28 / 4 = 7; same as Kannada version)
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Kuzushiji")    # ← CHANGED

MODELS_TF = {
    "WhatNet-Kuzushiji": lambda: build_whatnet_kuzushiji(NUM_CLASSES, IMG), # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)   # ceiling division
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [Kuzushiji-49]", "bold"))                                 # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results.json")   # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories.json") # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 benchmark complete.\n", "green", "bold")) # ← CHANGED

2026-04-19 21:53:46.739810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776635626.932710      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776635626.993671      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776635627.479191      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776635627.479237      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776635627.479241      55 computation_placer.cc:177] computation placer alr

[INFO] Loading train from split files: /kaggle/input/datasets/anokas/kuzushiji/k49-train-imgs.npz, /kaggle/input/datasets/anokas/kuzushiji/k49-train-labels.npz
[INFO] Loading test from split files: /kaggle/input/datasets/anokas/kuzushiji/k49-test-imgs.npz, /kaggle/input/datasets/anokas/kuzushiji/k49-test-labels.npz
[INFO] Unique classes in dataset : 49
[INFO] CFG num_classes           : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547
[INFO] RandomRotation available – rotation augmentation enabled (±8°).


I0000 00:00:1776635654.362584      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776635654.368890      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Kuzushiji-49]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Kuzushiji  –  Parameter Summary                     ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║              288  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_normali

I0000 00:00:1776635685.018299     129 service.cc:152] XLA service 0x7a9f3c003a20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776635685.018348     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776635685.018354     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776635689.633424     129 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776635712.190151     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/50  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=0.7742  acc=89.10%  val_loss=0.5984  val_acc=94.08%  lr=5.00e-04  [391.2s]
  Epoch   2/50  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.5374  acc=95.83%  val_loss=0.5303  val_acc=96.00%  lr=4.98e-04  [307.8s]
  Epoch   3/50  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.5038  acc=96.73%  val_loss=0.4881  val_acc=97.26%  lr=4.96e-04  [306.6s]
  Epoch   4/50  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.4832  acc=97.24%  val_loss=0.4750  val_acc=97.44%  lr=4.92e-04  [306.9s]
  Epoch   5/50  ██░░░░░░░░░░░░░░░░░░  10.0%  loss=0.4679  acc=97.63%  val_loss=0.4757  val_acc=97.39%  lr=4.88e-04  [306.0s]
  Epoch   6/50  ██░░░░░░░░░░░░░░░░░░  12.0%  loss=0.4572  acc=97.93%  val_loss=0.4658  val_acc=97.81%  lr=4.82e-04  [306.4s]
  Epoch   7/50  ██░░░░░░░░░░░░░░░░░░  14.0%  loss=0.4504  acc=98.14%  val_loss=0.4508  val_acc=98.19%  lr=4.76e-04  [306.1s]
  Epoch   8/50  ███░░░░░░░░░░░░░░░░░  16.0%  loss=0.4433  acc=98.29%  val_loss=0.4577  val_acc=97.93%  lr=4.69e-04  [305.7s]


# Method

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition
#  Adapted from the Kannada-MNIST version.
#
#  Transfer rationale
#  ──────────────────
#  Kuzushiji (くずし字) is classical cursive Japanese script used in pre-Meiji
#  documents.  Kuzushiji-49 covers 49 Hiragana character classes rendered in
#  this historical cursive style.  Key structural differences vs. Kannada:
#
#    • num_classes   : 10  → 49   (Kuzushiji-49: 49 Hiragana classes)
#    • image_size    : 28  → 28   (same – Kuzushiji-49 is natively 28×28)
#    • Dataset format: .npz arrays  (identical layout to Kannada-MNIST / MNIST)
#                      keys: arr_0 (images, uint8), arr_1 (labels, uint8)
#    • Dataset loader: npz loader retained from Kannada version (no changes)
#    • Scaffold branch: 1×3 (Kannada short horizontal) → 3×3 + 3×1 pair
#                       Kuzushiji characters combine flowing diagonal curves
#                       (captured by 3×3 isotropic) with explicit vertical
#                       stroke endings (captured by 3×1).  The horizontal 1×3
#                       is insufficient because Hiragana strokes are often
#                       diagonal, not axis-aligned.
#    • STM kernels   : 1×3/3×1 → 3×3 + 3×1 + two dilated rates (1 and 3)
#                       • 3×3 standard  – diagonal curves and stroke junctions
#                       • 3×1 vertical  – stroke endings / harai (払い) tails
#                       • dilation=1    – local curvature / stroke width
#                       • dilation=3    – character-level spatial context;
#                         Kuzushiji glyphs span the full 28-px frame so wider
#                         context is needed to separate visually similar chars
#                         (e.g. く vs. つ, き vs. さ)
#    • Augmentation  : random_flip REMOVED entirely – Kuzushiji characters
#                       have asymmetric strokes; flipping creates invalid glyphs
#                       (e.g. mirroring き would not produce any valid kana).
#                       random_rotation ±8° ADDED (via tfa or manual affine) –
#                       cursive documents vary greatly in writing angle, so
#                       mild rotation is the most important generalisation.
#                       NOTE: if tensorflow-addons is unavailable, rotation
#                       falls back to pad-crop only (a warning is printed).
#    • Model name    : WhatNet-Kannada → WhatNet-Kuzushiji
#    • Output files  : kannada_*.json  → kuzushiji_*.json
#
#  Everything else – architecture depth, LR schedule, callbacks,
#  display utilities – is unchanged.
#
#  Dataset
#  ───────
#  Kuzushiji-49  (Clanuwat et al., 2018  –  arXiv:1812.01718)
#    • 232 365 train + 38 547 test 28×28 greyscale images
#    • 49 classes (balanced-ish Hiragana subset of full Kuzushiji dataset)
#    • Download:
#        https://github.com/rois-codh/kmnist   (official – npz files)
#        https://www.kaggle.com/datasets/anokas/kuzushiji  (Kaggle mirror)
#    • Files expected in CFG['data_dir']:
#        k49-train-imgs.npz  –  arr_0: uint8 (N, 28, 28)
#        k49-train-lbls.npz  –  arr_0: uint8 (N,)
#        k49-test-imgs.npz   –  arr_0: uint8 (M, 28, 28)
#        k49-test-lbls.npz   –  arr_0: uint8 (M,)
#    • Alternative single-file layout (some mirrors):
#        k49_train.npz       –  keys: arr_0 (images), arr_1 (labels)
#        k49_test.npz        –  keys: arr_0 (images), arr_1 (labels)
#    • This script handles BOTH layouts automatically.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Kannada version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 49 classes – Kuzushiji Hiragana subset
    "num_classes":      49,
    # Same as Kannada-MNIST – Kuzushiji-49 is natively 28×28
    "image_size":       28,

    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # ← CHANGED: path to the Kuzushiji-49 .npz files.
    #   Official download (place *.npz files here):
    #     https://github.com/rois-codh/kmnist
    #   Kaggle mirror:
    #     https://www.kaggle.com/datasets/anokas/kuzushiji
    #   Kaggle notebook path:
    #     "/kaggle/input/kuzushiji"
    "data_dir":    "/kaggle/input/datasets/anokas/kuzushiji/",                          # ← CHANGED

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     ← CHANGED: Kuzushiji-49 ships as .npz files like Kannada-MNIST but uses
#     a SPLIT-FILE layout (separate image and label npz files) OR a combined
#     layout depending on the mirror used.  Both are handled automatically.
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download Kuzushiji-49 from:\n"
        "  https://github.com/rois-codh/kmnist  (official)\n"
        "  https://www.kaggle.com/datasets/anokas/kuzushiji  (Kaggle)\n"
        "Place the .npz files in the directory above."
    )

def _load_split_files(img_path: str, lbl_path: str):
    """
    Load from the official split-file layout:
      k49-train-imgs.npz  (arr_0 = uint8 images)
      k49-train-lbls.npz  (arr_0 = uint8 labels)
    """
    images = np.load(img_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(lbl_path)["arr_0"].astype(np.int32)    # (N,)
    return images, labels

def _load_combined_file(path: str):
    """
    Load from a single combined .npz:
      arr_0 = images (uint8), arr_1 = labels (uint8)
    """
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int32)
    return images, labels

def _load_kuzushiji(split: str):
    """
    Auto-detect layout (split-file vs combined) and load accordingly.
    split: 'train' or 'test'
    """
    d = CFG["data_dir"]
    # ── Layout A: official split files ──
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files: {img_path}, {lbl_path}")
        return _load_split_files(img_path, lbl_path)
    # ── Layout B: combined single file ──
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file: {combined}")
        return _load_combined_file(combined)
    # ── Not found ──
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data for Kuzushiji-49.\n"
        f"  Looked for (split-file): {img_path}, {lbl_path}\n"
        f"  Looked for (combined)  : {combined}\n"
        "Please verify the .npz files are in CFG['data_dir']."
    )

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes in dataset : {len(np.unique(y_train_all))}")
print(f"[INFO] CFG num_classes           : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,28,28) → (N,28,28,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Optional: mild rotation augmentation via tf.keras.layers.RandomRotation ───
# RandomRotation was introduced in TF 2.6. We detect it at runtime to stay
# compatible with older TF versions.
_HAS_RANDOM_ROTATION = hasattr(tf.keras.layers, "RandomRotation")
if _HAS_RANDOM_ROTATION:
    print("[INFO] RandomRotation available – rotation augmentation enabled (±8°).")
    _rotator = tf.keras.layers.RandomRotation(
        factor=8/360,          # ±8 degrees expressed as fraction of 2π
        fill_mode="constant",
        fill_value=-1.0,       # background fill after normalisation
        seed=SEED,
    )
else:
    print("[WARN] RandomRotation not available in this TF version – "
          "rotation augmentation disabled, using pad-crop only.")
    _rotator = None

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Stochastic augmentation – training only.

    Kuzushiji characters come from scanned historical documents with
    significant variation in:
      • Writing angle (cursive script written at various tilts)
      • Ink density / contrast
      • Slight spatial translation

    Augmentations:
      • Brightness / contrast jitter  – ink density variation
      • Pad-then-random-crop          – translation ±2 px
      • Random rotation ±8°           – writing angle variation  ← CHANGED
                                        (no flip – kana are asymmetric)

    No horizontal/vertical flip: mirroring any Kuzushiji glyph produces
    invalid or confusable characters (e.g. く ↔ mirrored-く which does not
    exist; or confusion between き and a non-standard form of さ).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    # ← CHANGED: apply mild rotation if available
    if _rotator is not None:
        img = _rotator(tf.expand_dims(img, 0))[0]
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Kuzushiji stroke topology encoder.                        ← CHANGED docstring

    Kuzushiji-49 (cursive historical Hiragana) has extremely diverse stroke
    patterns:
      • Flowing diagonal curves connecting multiple sub-strokes into one glyph
      • Explicit stroke endings (harai 払い – rightward fading diagonal tails)
      • High inter-class visual similarity: く/つ, き/さ, ぬ/め etc.
      • Glyph strokes span the full 28-px frame → wide spatial context needed

    Kernel design:
      • 3×3 standard    – diagonal curves, stroke junctions, general body
      • 3×1 vertical    – harai tail endpoints, descenders
      • dilation=1      – fine stroke width / curvature (local context)
      • dilation=3      – character-level context: spans ~13 px at 28-px
                           resolution, covering the full glyph height/width;
                           critical for resolving visually similar pairs

    No 1×N wide horizontal kernel: Kuzushiji strokes rarely run purely
    horizontal – diagonal is far more common.  A 3×3 base captures diagonal
    geometry better than axis-aligned asymmetric kernels.
    """
    # ← CHANGED: 3×3 + 3×1 + two dilation rates (1 and 3)
    base = layers.Conv2D(64, 3, padding="same", use_bias=False, activation=gelu)(x)
    v    = layers.Conv2D(32, (3, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1   = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d3   = layers.Conv2D(16, 3, padding="same", dilation_rate=3, use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([base, v, d1, d3])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed WhatNet-Kuzushiji; num_classes=49; image_size=28.
#     Scaffold uses 3×3 isotropic for diagonal Kuzushiji stroke geometry.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_kuzushiji(num_classes: int = 49, image_size: int = 28,
    drop_path_rate: float = 0.05,
    dropout_rate: float = 0.3,
    weight_decay: float = 1e-4,
    head_units: int = 512,
    override_tier: int = None) -> Model:
    """
    WhatNet-Kuzushiji: WhatNet adapted for Kuzushiji-49 Hiragana recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (glyph body)
      • Diagonal-stroke scaffold: 3×3 isotropic conv – captures diagonal
        cursive strokes that Kuzushiji glyphs are composed of (no axis-aligned
        headline like Bengali/Devanagari, no short cross-hair like Kannada)
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (14×14 at 28-px input)
      enc2: 64→128   ( 7× 7)
      enc3: 128→256  ( 4× 4, with padding)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB) – especially important here
        because fine strokes (d1) and global glyph shape (d3) need to interact
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 3×3 + 3×1 + dilation (rate 1 & 3)
      • Gated fusion → final MLP + layer norm → logits (49 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×3 (Kannada) → 3×3 (Kuzushiji diagonal strokes)
    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(7)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    # ← CHANGED pool factor: 8 → 7  (28 / 4 = 7; same as Kannada version)
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    # cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    # afc_in   = layers.GlobalAveragePooling2D()(enc3)
    # afc_in   = layers.Add()([afc_in, cstb_out])
    # afc_out  = adaptive_filter_capsule(afc_in, K)

    # stgm_out    = stroke_topology_module(enc3, 256)
    # combined    = layers.Concatenate()([stgm_out, afc_out])
    # gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    # stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    # fused       = layers.Concatenate()([stgm_scaled, afc_out])

    # x   = layers.Dense(512)(fused)
    # x   = layers.LayerNormalization()(x)
    # x   = layers.Activation(gelu)(x)
    # out = layers.Dense(K, name="logits")(x)
# ── Multi-scale GAP fusion ──────────────────────────────────────────────
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    # ── Adaptive Filter Capsule (AFC) ───────────────────────────────────────
    # Projects the fused multi-scale vector into capsule space.
    # Each of the K capsules learns to respond to one character class.
    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # ── Classification head ─────────────────────────────────────────────────
    # Dense projection of the raw GAP features (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # ── Gated fusion: AFC scores + dense-head logits ────────────────────────
    # A learned scalar gate (per-sample softmax over 2 weights) blends the
    # AFC capsule scores with the plain dense logits.  This lets the model
    # learn how much to trust the capsule routing vs. the direct projection.
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)

    # gate[:,0] weights the dense head; gate[:,1] weights the AFC output
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out,gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=outputs, name="WhatNet-Kuzushiji")    # ← CHANGED

MODELS_TF = {
    "WhatNet-Kuzushiji": lambda: build_whatnet_kuzushiji(NUM_CLASSES, IMG), # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)   # ceiling division
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [Kuzushiji-49]", "bold"))                                 # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
        # EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results.json")   # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories.json") # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 benchmark complete.\n", "green", "bold")) # ← CHANGED

2026-04-27 00:36:40.557617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777250200.946324      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777250201.070781      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777250202.024801      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777250202.024839      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777250202.024842      55 computation_placer.cc:177] computation placer alr

[INFO] Loading train from split files: /kaggle/input/datasets/anokas/kuzushiji/k49-train-imgs.npz, /kaggle/input/datasets/anokas/kuzushiji/k49-train-labels.npz
[INFO] Loading test from split files: /kaggle/input/datasets/anokas/kuzushiji/k49-test-imgs.npz, /kaggle/input/datasets/anokas/kuzushiji/k49-test-labels.npz
[INFO] Unique classes in dataset : 49
[INFO] CFG num_classes           : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547
[INFO] RandomRotation available – rotation augmentation enabled (±8°).


I0000 00:00:1777250246.791614      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777250246.797833      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [Kuzushiji-49]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Kuzushiji  –  Parameter Summary                     ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║              288  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_normal

I0000 00:00:1777250275.124647     127 service.cc:152] XLA service 0x7968f001ebc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777250275.124687     127 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777250275.124691     127 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777250279.134002     127 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777250300.425044     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3268/3268 ━━━━━━━━━━━━━━━━━━━━ 369s 98ms/step - accuracy: 0.7726 - loss: 1.5343 - val_accuracy: 0.9546 - val_loss: 0.8911
Epoch 2/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 301s 91ms/step - accuracy: 0.9574 - loss: 0.8582 - val_accuracy: 0.9676 - val_loss: 0.8280
Epoch 3/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 300s 91ms/step - accuracy: 0.9682 - loss: 0.8117 - val_accuracy: 0.9746 - val_loss: 0.8003
Epoch 4/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 300s 91ms/step - accuracy: 0.9733 - loss: 0.7898 - val_accuracy: 0.9778 - val_loss: 0.7801
Epoch 5/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 299s 91ms/step - accuracy: 0.9769 - loss: 0.7758 - val_accuracy: 0.9768 - val_loss: 0.7803
Epoch 6/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 299s 91ms/step - accuracy: 0.9791 - loss: 0.7672 - val_accuracy: 0.9789 - val_loss: 0.7727
Epoch 7/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 299s 91ms/step - accuracy: 0.9824 - loss: 0.7571 - val_accuracy: 0.9813 - val_loss: 0.7650
Epoch 8/100
3268/3268 ━━━━━━━━━━━━━━━━━━━━ 299s 91ms/step - accuracy: 0.98

# Method

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition  (PyTorch)
#  Converted from the TensorFlow/Keras version.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":     49,
    "image_size":      28,
    "batch_size":      64,
    "epochs":          100,
    "lr":              5e-4,
    "weight_decay":    1e-4,
    "label_smoothing": 0.1,
    "val_split":       0.1,
    "data_dir":        "/kaggle/input/datasets/anokas/kuzushiji/",
    "results_dir":     "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
def _load_split_files(img_path: str, lbl_path: str):
    images = np.load(img_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(lbl_path)["arr_0"].astype(np.int64)
    return images, labels

def _load_combined_file(path: str):
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int64)
    return images, labels

def _load_kuzushiji(split: str):
    d        = CFG["data_dir"]
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files.")
        return _load_split_files(img_path, lbl_path)
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file.")
        return _load_combined_file(combined)
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data.\n"
        f"  Checked: {img_path}, {lbl_path}, {combined}"
    )


class KuzushijiDataset(Dataset):
    """Wraps raw numpy arrays; applies transforms at __getitem__ time."""

    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None):
        # images: (N, 28, 28) float32  labels: (N,) int64
        self.images    = images[:, np.newaxis, :, :]  # → (N, 1, 28, 28)
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx])   # (1, 28, 28)
        lbl = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, lbl


# ── Normalise: [0,255] → [-1,1]
def normalise(img: torch.Tensor) -> torch.Tensor:
    return img / 127.5 - 1.0


# ── Augmentation pipeline (training only) ──────────────────────────────────
#   • Brightness / contrast jitter
#   • Pad 2 px → random 28×28 crop  (≈ ±2 px translation)
#   • Random rotation ±8°
#   No horizontal / vertical flip  (asymmetric Kuzushiji glyphs)
train_transform = transforms.Compose([
    transforms.Lambda(normalise),
    transforms.Lambda(lambda x: x.expand(3, -1, -1)),          # need 3-ch for ColorJitter
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Lambda(lambda x: x.mean(0, keepdim=True)),       # back to 1-ch
    transforms.Pad(2, fill=-1.0),
    transforms.RandomCrop(IMG),
    transforms.RandomRotation(degrees=8, fill=-1.0),
])

val_transform = transforms.Compose([
    transforms.Lambda(normalise),
])


# ── Load & split ──────────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes : {len(np.unique(y_train_all))}")

rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(x_train_all))
x_train_all = x_train_all[perm]
y_train_all = y_train_all[perm]

n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

train_ds   = KuzushijiDataset(x_train, y_train, transform=train_transform)
val_ds     = KuzushijiDataset(x_val,   y_val,   transform=val_transform)
test_ds    = KuzushijiDataset(x_test,  y_test,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

# ─────────────────────────────────────────────────────────────────────────────
#  3. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    """Two 3×3 conv + BN + GELU with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.net(x) + x)


class DenseResBlock(nn.Module):
    """
    Three chained residual blocks whose outputs are concatenated, then fused
    with a 1×1 conv and downsampled 2× via depthwise-separable strided conv.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        # projection when channel counts differ
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            )
        else:
            self.proj = nn.Identity()

        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)

        # fuse 3×out_ch → out_ch, then stride-2 depthwise-sep
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.down = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                      groups=out_ch, bias=False),   # depthwise
            nn.Conv2d(out_ch, out_ch, 1, bias=False),  # pointwise
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        x = self.proj(x)
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = self.fuse(cat)
        out = self.down(out)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, H, W)
        w = x.mean(dim=[2, 3])         # GAP → (B, C)
        w = self.fc(w).unsqueeze(-1).unsqueeze(-1)   # (B, C, 1, 1)
        return x * w


class AdaptiveFilterCapsule(nn.Module):
    """
    Maps a flat feature vector to K capsule scores, one per class.
    Each capsule learns a class-specific filter over a low-dim projection.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 16):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.h = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Linear(256, num_classes * capsule_dim),
        )
        self.x_proj = nn.Linear(in_features, capsule_dim)
        self.bn     = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B  = x.size(0)
        h  = self.h(x).view(B, self.num_classes, self.capsule_dim)   # (B,K,D)
        xp = self.x_proj(x).unsqueeze(1).expand_as(h)                # (B,K,D)
        caps = (h * xp).sum(dim=-1)                                   # (B,K)
        caps = self.bn(caps)
        return caps


class StrokeTopologyModule(nn.Module):
    """
    Kuzushiji stroke topology encoder.
      • 3×3 standard (diagonal curves, junctions)
      • 3×1 vertical (harai tails, stroke endings)
      • dilation=1   (local curvature)
      • dilation=3   (glyph-level context; ~13 px receptive field)
    """
    def __init__(self, in_ch: int, out_features: int):
        super().__init__()
        self.base = nn.Conv2d(in_ch, 64, 3, padding=1, bias=False)
        self.vert = nn.Conv2d(in_ch, 32, (3, 1), padding=(1, 0), bias=False)
        self.d1   = nn.Conv2d(in_ch, 32, 3, padding=1,  dilation=1, bias=False)
        self.d3   = nn.Conv2d(in_ch, 16, 3, padding=3,  dilation=3, bias=False)
        self.bn   = nn.BatchNorm2d(64 + 32 + 32 + 16)
        self.fc   = nn.Linear(64 + 32 + 32 + 16, out_features)

    def forward(self, x):
        base = F.gelu(self.base(x))
        vert = F.gelu(self.vert(x))
        d1   = F.gelu(self.d1(x))
        d3   = F.gelu(self.d3(x))
        cat  = torch.cat([base, vert, d1, d3], dim=1)
        cat  = self.bn(cat)
        gap  = cat.mean(dim=[2, 3])            # GAP
        out  = F.gelu(self.fc(gap))
        return out


class CrossScaleTransformerBridge(nn.Module):
    """Multi-head attention across three encoder-scale summaries."""
    def __init__(self, dim: int = 256, num_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.ln   = nn.LayerNorm(dim)
        self.fc   = nn.Linear(dim, dim)

    def forward(self, s1_gap, s2_gap, s3_gap):
        # s*_gap: (B, dim)  →  seq: (B, 3, dim)
        seq = torch.stack([s1_gap, s2_gap, s3_gap], dim=1)
        out, _ = self.attn(seq, seq, seq)
        out = self.ln(out + seq)
        pooled = out.mean(dim=1)               # (B, dim)
        return F.gelu(self.fc(pooled))

# ─────────────────────────────────────────────────────────────────────────────
#  4. MODEL
# ─────────────────────────────────────────────────────────────────────────────

class WhatNetKuzushiji(nn.Module):
    """
    WhatNet adapted for Kuzushiji-49 Hiragana recognition.

    Stem (dual-path):
      • Standard 3×3 conv   – glyph body
      • Scaffold 3×3 conv   – diagonal cursive stroke structure
      → Concatenated, refined with SE channel attention

    Encoder (3 dense-res stages, each ×2 spatial downsampling):
      enc1: 64  → 64   (14×14)
      enc2: 64  → 128  ( 7× 7)
      enc3: 128 → 256  ( 4× 4, approx)
      Weighted scaffold residuals (+0.1) injected at each stage.

    Decoder head:
      • Multi-scale GAP fusion (enc1 || enc2 || enc3)
      • Adaptive Filter Capsule (AFC)
      • Dense head (linear → LN → GELU → logits)
      • Learned 2-way gate blends AFC and dense-head logits
    """

    def __init__(self, num_classes: int = 49, image_size: int = 28,
                 dropout_rate: float = 0.3, head_units: int = 512):
        super().__init__()
        K = num_classes

        # ── Stem ──────────────────────────────────────────────────────────
        self.stem_main = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_scaffold = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_se   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
        )

        # ── Scaffold projections (for residual injection) ──────────────────
        self.sc_proj1 = nn.Conv2d(32, 64,  1, bias=False)
        self.sc_proj2 = nn.Conv2d(32, 128, 1, bias=False)
        self.sc_proj3 = nn.Conv2d(32, 256, 1, bias=False)

        # ── Encoder ───────────────────────────────────────────────────────
        self.enc1 = DenseResBlock(64,  64)
        self.enc2 = DenseResBlock(64,  128)
        self.enc3 = DenseResBlock(128, 256)

        # ── Decoder head ──────────────────────────────────────────────────
        fused_dim = 64 + 128 + 256    # 448
        self.afc      = AdaptiveFilterCapsule(fused_dim, K)
        self.head_fc  = nn.Linear(fused_dim, head_units, bias=False)
        self.head_ln  = nn.LayerNorm(head_units)
        self.head_out = nn.Linear(head_units, K)
        self.gate_fc  = nn.Linear(K + K, 2)   # gate over (dense || afc)

    def forward(self, x):
        # x: (B, 1, 28, 28)

        # ── Stem ──────────────────────────────────────────────────────────
        t        = self.stem_main(x)          # (B, 32, 28, 28)
        scaffold = self.stem_scaffold(x)      # (B, 32, 28, 28)
        stem = torch.cat([t, scaffold], dim=1)  # (B, 64, 28, 28)
        stem = self.stem_se(stem)
        stem = self.stem_proj(stem)

        # ── Encoder with scaffold residuals ───────────────────────────────
        # enc1: (B, 64, 14, 14)
        enc1 = self.enc1(stem)
        sc1  = F.avg_pool2d(self.sc_proj1(scaffold), 2)
        enc1 = enc1 + 0.1 * sc1

        # enc2: (B, 128, 7, 7)
        enc2 = self.enc2(enc1)
        sc2  = F.avg_pool2d(self.sc_proj2(scaffold), 4)
        enc2 = enc2 + 0.1 * sc2

        # enc3: (B, 256, 4, 4)  (28 → 14 → 7 → 3 or 4 depending on padding)
        enc3 = self.enc3(enc2)
        sc3  = F.adaptive_avg_pool2d(self.sc_proj3(scaffold),
                                     enc3.shape[-2:])
        enc3 = enc3 + 0.1 * sc3

        # ── Multi-scale GAP fusion ─────────────────────────────────────────
        gap1 = enc1.mean(dim=[2, 3])           # (B, 64)
        gap2 = enc2.mean(dim=[2, 3])           # (B, 128)
        gap3 = enc3.mean(dim=[2, 3])           # (B, 256)
        fused = torch.cat([gap1, gap2, gap3], dim=1)  # (B, 448)

        # ── AFC ───────────────────────────────────────────────────────────
        afc_out = self.afc(fused)              # (B, K)

        # ── Dense head ────────────────────────────────────────────────────
        h = F.gelu(self.head_ln(self.head_fc(fused)))
        dense_out = self.head_out(h)           # (B, K)

        # ── Gated fusion ──────────────────────────────────────────────────
        gate = F.softmax(
            self.gate_fc(torch.cat([dense_out, afc_out], dim=1)),
            dim=1,
        )                                      # (B, 2)
        logits = dense_out * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return logits                          # (B, K)


# ─────────────────────────────────────────────────────────────────────────────
#  5. LR SCHEDULE  –  Cosine annealing without restarts
# ─────────────────────────────────────────────────────────────────────────────
def cosine_annealing_lr(optimizer, step: int, total_steps: int, base_lr: float,
                        min_lr: float = 1e-6) -> float:
    cosine = 0.5 * (1.0 + np.cos(np.pi * step / total_steps))
    lr     = max(base_lr * cosine, min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


# ─────────────────────────────────────────────────────────────────────────────
#  6. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"


def print_model_summary(model: nn.Module) -> None:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    W         = 62
    title     = f"  {model.__class__.__name__}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {total-trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))


def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    step: int, total_steps: int) -> tuple[float, float, int]:
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)

        lr = cosine_annealing_lr(optimizer, step, total_steps, CFG["lr"])
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                            enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
        step       += 1

    return total_loss / n, correct / n, step


@torch.no_grad()
def evaluate(model, loader, criterion) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        logits     = model(imgs)
        loss       = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def compute_macro_f1(model, loader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for imgs, lbls in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        preds = model(imgs).argmax(1).cpu().numpy()
        lbls  = lbls.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. MAIN TRAINING RUN
# ─────────────────────────────────────────────────────────────────────────────

model     = WhatNetKuzushiji(NUM_CLASSES, IMG).to(DEVICE)
print_model_summary(model)

# Label-smoothing cross-entropy (from_logits – no softmax in model)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

best_val_acc  = 0.0
patience_ctr  = 0
PATIENCE      = 15
ckpt_path     = os.path.join(CFG["results_dir"], "WhatNet-Kuzushiji_best.pt")

step      = 0
all_hist  = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

t_start = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    tr_loss, tr_acc, step = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, step, total_steps)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)

    all_hist["loss"].append(tr_loss)
    all_hist["accuracy"].append(tr_acc)
    all_hist["val_loss"].append(val_loss)
    all_hist["val_accuracy"].append(val_acc)

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(
        f"  {_c(f'Epoch {epoch:>3}/{CFG[\"epochs\"]}', 'grey')}  "
        f"{_c(f'loss={tr_loss:.4f}', 'white')}  "
        f"{_c(f'acc={tr_acc*100:.2f}%', 'green')}  "
        f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
        f"{_c(f'val_acc={val_acc*100:.2f}%', 'yellow' if val_acc < tr_acc else 'green')}  "
        f"{_c(f'lr={lr_now:.2e}', 'blue')}  "
        f"{_c(f'[{elapsed:.1f}s]', 'grey')}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {epoch} (patience={PATIENCE})", "yellow"))
            break

elapsed_total = time.time() - t_start
print(f"\n  {_c('✔ Done:', 'green', 'bold')} best val acc = "
      f"{_c(f'{best_val_acc*100:.2f}%', 'green')}  "
      f"wall time = {_c(f'{elapsed_total:.0f}s', 'grey')}")

# ── Reload best checkpoint for evaluation ────────────────────────────────────
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc_raw = evaluate(model, test_loader, criterion)
test_acc  = test_acc_raw * 100.0
macro_f1  = compute_macro_f1(model, test_loader)
n_params  = sum(p.numel() for p in model.parameters())

results = {
    "WhatNet-Kuzushiji": {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    n_params,
        "test_loss": round(float(test_loss), 4),
    }
}
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {k: [float(v) for v in vs] for k, vs in all_hist.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 benchmark complete.\n", "green", "bold"))